# Exercise 1.5.3.15 — fill in the missing code below

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `1.5.3 OthelloGPT`  
**Notebook:** `1.5.3_OthelloGPT_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=1.5.3.15](https://delta-drills.vercel.app/?arena_exercise=1.5.3.15)


# [1.5.3] OthelloGPT (exercises)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/33_[1.5.3]_OthelloGPT)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part53_othellogpt/1.5.3_OthelloGPT_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part53_othellogpt/1.5.3_OthelloGPT_solutions.ipynb?t=20260329)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-15-3.png" width="350">

# Introduction

*Note - unless otherwise specified, first person here refers to the primary researcher, Neel Nanda.*

[Emergent World Representations](https://arxiv.org/pdf/2210.13382) is a fascinating recent ICLR Oral paper from Kenneth Li et al, summarised in [Kenneth's excellent post on the Gradient](https://thegradient.pub/othello/). They trained a model (Othello-GPT) to play legal moves in the board game Othello, by giving it random games (generated by choosing a legal next move uniformly at random) and training it to predict the next move. The headline result is that Othello-GPT learns an emergent world representation - despite never being explicitly given the state of the board, and just being tasked to predict the next move, it learns to compute the state of the board at each move. (Note that the point of Othello-GPT is to play legal moves, not good moves, though they also study a model trained to play good moves.)

They present two main pieces of evidence. They can extract the board state from the model's residual stream via non-linear probes (a two layer ReLU MLP). And they can use the probes to causally intervene and change the model's representation of the board (by using gradient descent to have the probes output the new board state) - the model now makes legal moves in the new board state even if they are not legal in the old board, and even if that board state is impossible to reach by legal play!

I've strengthened their headline result by finding that much of their more sophisticated (and thus potentially misleading) techniques can be significantly simplified. Not only does the model learn an emergent world representation, it learns a linear emergent world representation, which can be causally intervened on in a linear way! But rather than representing "this square has a black/white piece", it represents "this square has my/their piece". The model plays both black and white moves, so this is far more natural from its perspective. With this insight, the whole picture clarifies significantly, and the model becomes far more interpretable!

### The research process

You can read more about the research process in [this post here](https://www.lesswrong.com/s/nhGNHyJHbrofpPbRG/p/TAz44Lb9n9yf52pv8), which we'd strongly recommend. The exercises are structured in a different way to the chronological research process (e.g. we look at probes early on, when actually training a probe is a high-effort thing and if you were trying to get traction on a problem like this you'd want to use more basic techniques first, like logit lens / attention pattern analysis).

As you're going through these exercises, we encourage you to keep thinking about how you would approach this research process. Do any of the results / approaches seem completely out of the blue for you, and if so can you think about what the justification might have been for trying them? What would you have tried first, and why?

### How you should approach these exercises

There's a lot of setup code for analysing OthelloGPT (this is somewhat unavoidable). There are also a lot of different plots of tensors with different dimensions and different meanings, and it can be hard to keep track of. You don't have to keep track of all of this in your head, but **we strongly recommend explaining to yourself or your pair programming partner what the significance of each result is before you move on to the next section.** Sometimes exercises and questions will prompt you to do this, but you should make it a reflex!

At the end of each subsection in the first section (which mostly consists of setup), there's a recap of all the objects we've defined, why they matter, and how they can be used. There will also be a "recap of this section" at the end of each of the later sections, where we review the key results and their significance.

## The purpose / structure of these exercises

At a surface level, these exercises are designed to help you understand the OthelloGPT model and the forms of probing & analysis that were done on it. But it's also designed to make you a better interpretability researcher! As a result, most exercises will be doing a combination of:

1. Showing you some new feature/component of OthelloGPT, and
2. Teaching you how to use tools and interpret results in a broader mech interp context.

As you're going through these exercises, it's easy to get lost in the fiddly details of the techniques you're implementing or the things you're computing. Make sure you keep taking a high-level view, asking yourself what questions you're currently trying to ask and how you'll interpret the output you're getting, as well as how the tools you're currently using are helping guide you towards a better understanding of the model.

## Content & Learning Objectives

### 1️⃣ Model Setup & Linear Probes

In this section, we'll set up the model that we'll use for the rest of the exercises, and load in weights. We'll also get familiar with the datasets and objects that we'll be using for this analysis. Lastly, we'll learn about **linear probes** and how they are used.

> ##### Learning Objectives
>
> - Understand the basic structure of the Othello-GPT model
> - Become familiar with the basic tools we'll use for board visualization during these exercises
> - See how our linear probe works, and how we can change basis between "black/white piece" and "mine/their piece"
> - Intervene with the linear probe to influence model predictions

### 2️⃣ Looking for modular circuits

Here, we'll use our probe to start to analyse circuits in our model. We can apply them to our neurons' output weights to identify which neurons matter and in what way, and we can also figure out when and where information is represented in a model.

> ##### Learning Objectives
>
> - Learn how to use our linear probe across multiple layers
> - Apply activation patching at a given sequence position to test hypotheses about our model
> - Understand how a neuron can be characterized in terms of its input and output weights

### 3️⃣ Neuron Interpretability: A Deep Dive

To practice neuron interpretability, we'll take a deep dive into understanding one particular neuron - the techniques and code should transfer pretty well to any other neurons!

The spirit of this section is to practice doing various standard things that you could go and apply to another neuron in practice - I end it still being fairly confused, and with many dangling threads!

> ##### Learning Objectives
>
> - Apply **direct logit attribution** to understand how a neuron's output weights directly effect predictions
> - Use SVD-based techniques to assess how much of a neuron's input/output behaviour is captured by some subspace
> - Use techniques like **max activating datasets** and **spectrum plots**, and understand their strengths and limitations

### 4️⃣ Training a Probe

In this section, we'll look at how to actually train our linear probe. This section is less mechanistic interpretability and more standard ML techniques, but it's still important to understand how to do this if you want to do any analysis like this yourself!

> ##### Learning Objectives
>
> - Learn how to set up and train a linear probe
> - See how to train multiple probes at once, and log their performance to Weights & Biases

### ☆ Bonus

Finally, we'll take a look at some future directions which could come from this OthelloGPT analysis. We strongly recommend you follow some of these threads of research yourself, and see where they might lead!

## Setup code

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install einops eindex-callum jaxtyping wandb transformer_lens==2.17.0 "git+https://github.com/neelnanda-io/neel-plotly"

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import copy
import os
import sys
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Callable

import einops
import numpy as np
import pandas as pd
import plotly.express as px
import torch as t
import wandb
from eindex import eindex
from jaxtyping import Bool, Float, Int
from torch import Tensor
from tqdm.notebook import tqdm
from transformer_lens import ActivationCache, HookedTransformer, HookedTransformerConfig
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import download_file_from_hf, get_act_name, to_numpy

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part53_othellogpt"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part53_othellogpt.tests as tests
import part53_othellogpt.utils as utils
from neel_plotly import scatter

t.set_grad_enabled(False)

MAIN = __name__ == "__main__"

# 1️⃣ Model Setup & Linear Probes

> ##### Learning Objectives
>
> - Understand the basic structure of the Othello-GPT model
> - Become familiar with the basic tools we'll use for board visualization during these exercises
> - See how our linear probe works, and how we can change basis between "black/white piece" and "mine/their piece"
> - Intervene with the linear probe to influence model predictions

For those unfamiliar, Othello is a board game analogous to chess or go, with two players, black and white, see the rules outlined in the figure below. I found [playing the AI on eOthello](https://www.eothello.com/) helpful for building intuition. A single move can change the colour of pieces far away (so long as there's a continuous vertical, horizontal or diagonal line), which means that calculating board state is actually pretty hard! (to my eyes much harder than in chess)

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/nmxzr2zsjNtjaHh7x/eoi5u5fodorjswnmbqk7" width="700">

## Background - Othello, and OthelloGPT

But despite the model just needing to predict the next move, it spontaneously learned to compute the full board state at each move - a fascinating result. A pretty hot question right now is whether LLMs are just bundles of statistical correlations or have some real understanding and computation! This gives suggestive evidence that simple objectives to predict the next token can create rich emergent structure (at least in the toy setting of Othello). Rather than just learning surface level statistics about the distribution of moves, it learned to model the underlying process that generated that data. In my opinion, it's already pretty obvious that transformers can do something more than statistical correlations and pattern matching, see eg [induction heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html), but it's great to have clearer evidence of fully-fledged world models!

For context on my investigation, it's worth analysing exactly the two pieces of evidence they had for the emergent world representation, the probes and the causal interventions, and their strengths and weaknesses.

### Probes

The probes give suggestive, but far from conclusive evidence. When training a probe to extract some feature from a model, it's easy to trick yourself. It's crucial to track whether the probe is just reading out the feature, or actually computing the feature itself, and reading out much simpler features from the model. In the extreme case, you could attach a much more powerful model as your "probe", and have it just extract the input moves, and then compute the board state from scratch! They found that linear probes did not work to recover board state (with an error rate of 20.4%): (ie, projecting the residual stream onto some 3 learned directions for each square, corresponding to empty, black and white logits). While the simplest non-linear probes (a two layer MLP with a single hidden ReLU layer) worked extremely well (an error rate of 1.7%). Further (as described in their table 2, screenshot below), these non-linear probes did not work on a randomly initialised network, and worked better on some layers than others, suggesting they were learning something real from the model.

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/nmxzr2zsjNtjaHh7x/b5rxfns1wxvsuomh1d79" width="550">

### Causal interventions

Probes on their own can mislead, and don't necessarily tell us that the model uses this representation - the probe could be extracting some vestigial features or a side effect of some more useful computation, and give a misleading picture of how the model computes the solution. But their causal interventions make this much more compelling evidence. They intervene by a fairly convoluted process (detailed in the figure below, though you don't need to understand the details), which boils down to choosing a new board state, and applying gradient descent to the model's residual stream such that our probe thinks the model's residual stream represents the new board state. I have an immediate skepticism of any complex technique like this: when applying a powerful method like gradient descent it's so easy to wildly diverge from what the models original functioning is like! But the fact that the model could do the non-trivial computation of converting an edited board state into a legal move post-edit is a very impressive result! I consider it very strong evidence both that the probe has discovered something real, and that the representation found by the probe is causally linked to the model's actual computation!

## Naive Implications for Mechanistic Interpretability

I was very interested in this paper, because it simultaneously had the fascinating finding of an emergent world model (and I'm also generally into any good interp paper), yet something felt off. The techniques used here seemed "too" powerful. The results were strong enough that something here seemed clearly real, but my intuition is that if you've really understood a model's internals, you should just be able to understand and manipulate it with far simpler techniques, like linear probes and interventions, and it's easy to be misled by more powerful techniques.

In particular, my best guess about model internals is that the networks form [**decomposable, linear representations**](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=KiDMIteKCEXt_EkR2sZCOfbG): that the model computes a bunch of useful features, and represents these as directions in activation space. See [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html#motivation) for some excellent exposition on this. This is decomposable because each feature can vary independently (from the perspective of the model - on the data distribution they're likely dependent), and linear because we can extract a feature by projecting onto that feature's direction (if the features are orthogonal - if we have something like [superposition](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=3br1psLRIjQCOv2T4RN3V6F2) it's messier). This is a natural way for models to work - they're fundamentally a series of matrix multiplications with some non-linearities stuck in convenient places, and a decomposable, linear representation allows it to extract any combination of features with a linear map!

Under this framework, if a feature can be found by a linear probe then the model has already computed it, and if that feature is used in a circuit downstream, we should be able to causally intervene with a linear intervention, just changing the coordinate along that feature's direction. So the fascinating finding that linear probes do not work, but non-linear probes do, suggests that either the model has a fundamentally non-linear representation of features (which it is capable of using directly for downstream computation!), or there's a linear representation of simpler and more natural features, from which the probe computes board state. My prior was on a linear representation of simpler features, but the causal intervention findings felt like moderate evidence for the non-linear representation. And the non-linear representation hypothesis would be a big deal if true! If you want to reverse-engineer a model, you need to have a crisp picture of how its computation maps onto activations and weights, and this would break a lot of my beliefs about how this correspondance works! Further, linear representations are just really *convenient* to reverse-engineer, and this would make me notably more pessimistic about mechanistic interpretability working.

## Model Setup & Visualization

The best way to become less confused about a mysterious model behaviour is to mechanistically analyse it. To zoom in on whatever features and circuits we can find, build our understanding from the bottom up, and use this to form grounded beliefs about what's actually going on.

To get started, let's load in the model that we'll be using for this chapter. A few important technical details:

- Othello games have 60 moves (the board has size 8x8, and the middle 4 squares are all occupied, so by 60 moves all squares are occupied by some color). The model's context length is 59, because it's learned to take in `moves[0:59]` and predict `moves[1:60]`. 
- The dataset the model was trained on just consists of randomly sampled legal moves. A move in Othello is legal if and only if it captures the opponents' pieces in a horizontal, vertical or diagonal line - in which case it flips all those pieces to its own color. This means **predicting next legal moves is a nontrivial task (one might guess it requires the mdoel to track captures and board states over time)**
    - Because the model was trained on cross entropy loss against the actual next moves, the model's distribution converges to uniform over all next legal moves - can you see why?

<details>
<summary>Proof - why the model learns a uniform distribution over next legal moves</summary>

The cross entropy loss is defined as $H(p, q) = -\sum_x p(x) \log q(x)$, which by [Gibbs' inequality](https://en.wikipedia.org/wiki/Gibbs%27_inequality) is minimized when $p(x) = q(x)$. So we minimize loss when the model's distribution $q$ matches the underlying data distribution $p$, which is uniform over next legal moves.

Note we can also prove this using Jensen's inequality, since $\log q(x)$ is a strictly concave function. Alternatively, we can express the cross entropy loss as the sum of the underlying data distribution's entropy and the KL divergence between the model's distribution and the underlying distribution:

$$
H(p, q) = -\sum_x p(x) \log q(x) = \sum_x p(x) \log \frac{p(x)}{q(x)} - \sum_x p(x) \log p(x) = D_{\mathrm{KL}}(p \| q) + H(p)
$$

so this statement becomes equivalent to the statement that $D_{\mathrm{KL}}(p \| q)$ is always non-negative and is only zero when $p(x) = q(x)$ (which can be proved in similar ways).

</details>

- The vocabulary size is 61, because we allow any of the 8x8 - 4 = 60 unoccupied squares to be played, plus the pass move. The vocab is ordered `pass, A0, A1, ..., H7`. Note that we'll be filtering out games where a `pass` move was played, so we don't need to worry about this.
- We'll refer to squares in 3 different ways: 
    1. **label** - this is the string representation, i.e. `"pass"`, `"A0"`, `"A1"`, ..., `"H7"`.
    2. **token id**, or **id** - this is the token ID in the model's vocab, i.e. `1` for A0, ..., `60` for H7. We skip `0` which is the token id for `pass`, and we skip the 4 middle squares since they're always occupied and so there are no moves in or predictions for these squares.
    3. **square index**, or **square** - this is the zero-indexed value of the square in the size-64 board, i.e. `0` for A0, `1` for A1, ..., `63` for H7.
- Black plays first in Othello, and so (in games with no passes) White plays last. Since we don't predict the very first move, this means the model's predictions are for (white 1, black 2, white 2, ..., white 29, black 30, white 30).


<!-- Here are some more files, which we won't be using directly in these exercises, but you might still like to have a look at:

* `tl_probing.py` is my probe training file. But it was used to train a second probe, linear_probe_L4_blank_vs_color_v1.pth . This probe actually didn't work very well for analysing the model (despite getting great accuracy) and I don't know why - it was trained on layer 4, to do a binary classification on blank vs not blank, and on my color vs their color *conditional *on not being blank (ie not evaluated if blank). For some reason, the "this cell is my color" direction has a significant dot product with the "is blank" direction, and this makes it much worse for e.g. interpreting neurons. I don't know why!
* `tl_scratch.py` is where I did some initial exploration, including activation patching between different final moves.
* `tl_exploration.py` is where I did my most recent exploration, verifying that the probe works, doing probe interventions (CTRL F for `newly_legal`) and using the probe to interpret neurons. -->

Run the code below, to load in the model:

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=8,
    d_model=512,
    d_head=64,
    n_heads=8,
    d_mlp=2048,
    d_vocab=61,
    n_ctx=59,
    act_fn="gelu",
    normalization_type="LNPre",
    device=device,
)
model = HookedTransformer(cfg)

state_dict_synthetic = download_file_from_hf("NeelNanda/Othello-GPT-Transformer-Lens", "synthetic_model.pth")
# state_dict_championship = download_file_from_hf("NeelNanda/Othello-GPT-Transformer-Lens", "championship_model.pth")

model.load_state_dict(state_dict_synthetic)

Next, run the code below to check that the model is working correctly. Our `sample_input` is a sequence of 10 moves (the first 10 moves of the game, starting with black's 1st move and ending in white's 5th move).

In [ ]:
# An example input: 10 moves in a game
sample_input = t.tensor([[20, 19, 18, 10, 2, 1, 27, 3, 41, 42]]).to(device)

logits = model(sample_input)
logprobs = logits.log_softmax(-1)

assert logprobs.shape == (1, 10, 61)  # shape is [batch, seq_len, d_vocab]
assert logprobs[0, 0].topk(3).indices.tolist() == [
    21,
    33,
    19,
]  # these are the 3 legal moves, as we'll soon show

Note that we can convert each of our logprob vectors (which have shape `(61,)`) into a board state tensor of shape `(8, 8)` (we put very negative values in the 4 middle squares) via some clever indexing. Then, we can plot the results using a helper function `utils.plot_board_values` we've written for you. Since you'll be using this function a lot in subsequent exercises, we'll go through the important arguments here:

- `state`: a tensor of shape `(*N, 8, 8)`, where `*N` is any number of batch dimensions. For each `N` tensors, we'll visualize a single 8x8 grid. If the shape is `(8, 8)` then we'll plot a single board.
- `board_titles`: if plotting multiple boards (i.e. `N > 1`) then we can use this argument to label each of them.
- `boards_per_row`: if plotting multiple boards, we can set this many boards per row.
- `text`: an optional list of strings with the same shape as `state`, which we can use to annotate the boards. Also fine for it to be broadcastable to the shape of `state` (e.g. it can be `(8, 8)` even if `state` is 3D).
- `kwargs`: other arguments get passed into `px.imshow` (e.g. `title`, `width` and `height`)

Understanding exactly how this function works isn't important, since you can always copy and paste code from one of the following examples if you forget how to use it!

In [ ]:
MIDDLE_SQUARES = [27, 28, 35, 36]
ALL_SQUARES = [i for i in range(64) if i not in MIDDLE_SQUARES]

logprobs_board = t.full(size=(8, 8), fill_value=-13.0, device=device)
logprobs_board.flatten()[ALL_SQUARES] = logprobs[0, 0, 1:]  # the [1:] is to filter out logits for the "pass" move

utils.plot_board_values(logprobs_board, title="Example Log Probs", width=500)

<details>
<summary>Aside - how does this tensor indexing magic work?</summary>

`temp_board_state` is an array of shape `(8, 8)`. When we use `.flatten()`, this returns a **view** (i.e. same underlying data) with shape `(64,)`. When we index it by `ALL_SQUARES` (a list of 60 indices, which is all the indices excluding the "middle squares"), this also returns a view (still the same data). We can then set those 60 elements to be the model's log probs. This will change the values in the original tensor, without changing that tensor's shape.
</details>

We can use the `text` argument to annotate all the legal squares with their token IDs, which might be easier in some cases. Here's an example (you might want to reuse this code in some later exercises):

In [ ]:
TOKEN_IDS_2D = np.array([str(i) if i in ALL_SQUARES else "" for i in range(64)]).reshape(8, 8)
BOARD_LABELS_2D = np.array(["ABCDEFGH"[i // 8] + f"{i % 8}" for i in range(64)]).reshape(8, 8)

print(TOKEN_IDS_2D)
print(BOARD_LABELS_2D)

utils.plot_board_values(
    t.stack([logprobs_board, logprobs_board]),  # shape (2, 8, 8)
    title="Example Log Probs (with annotated token IDs)",
    width=800,
    text=np.stack([TOKEN_IDS_2D, BOARD_LABELS_2D]),  # shape (2, 8, 8)
    board_titles=["Labelled by token ID", "Labelled by board label"],
)

We can also use this function to plot multiple board states at once, exactly the same way. Again pay attention to the indexing, because understanding how this works will be useful going forwards!

In [ ]:
logprobs_multi_board = t.full(size=(10, 8, 8), fill_value=-13.0, device=device)
logprobs_multi_board.flatten(1, -1)[:, ALL_SQUARES] = logprobs[0, :, 1:]  # we now do all 10 moves at once

utils.plot_board_values(
    logprobs_multi_board,
    title="Example Log Probs",
    width=1000,
    boards_per_row=5,
    board_titles=[f"Logprobs after move {i}" for i in range(1, 11)],
)

Let's use the same function to plot the first 10 board states, and check that the predictions shown above make sense given the board state at that time. We'll use a helper method `OthelloBoardState` to track the board state after each move.

In [ ]:
board_states = t.zeros((10, 8, 8), dtype=t.int32)
legal_moves = t.zeros((10, 8, 8), dtype=t.int32)

board = utils.OthelloBoardState()
for i, token_id in enumerate(sample_input.squeeze()):
    # board.umpire takes a square index (i.e. from 0 to 63) and makes a move on the board
    board.umpire(utils.id_to_square(token_id))

    # board.state gives us the 8x8 numpy array of 0 (blank), -1 (black), 1 (white)
    board_states[i] = t.from_numpy(board.state)

    # board.get_valid_moves() gives us a list of the indices of squares that are legal to play next
    legal_moves[i].flatten()[board.get_valid_moves()] = 1

# Turn `legal_moves` into strings, with "o" where the move is legal and empty string where illegal
legal_moves_annotation = np.where(to_numpy(legal_moves), "o", "").tolist()

utils.plot_board_values(
    board_states,
    title="Board states",
    width=1000,
    boards_per_row=5,
    board_titles=[f"State after move {i}" for i in range(1, 11)],
    text=legal_moves_annotation,
)

In this plot you should see that each board state evolves from the previous one via a single move that captures a set of pieces from the opposing color (e.g. in the first move black plays `C3` capturing white at `D3`, and in the second move white plays `C2` and captures back black diagonally at `D3`). You should also see that the annotated legal moves match the moves predicted by the model.

## Data

Let's now load some data for OthelloGPT. We'll load data in in `id` format (i.e. 1 to 60 inclusive, since our vocab is `range(0, 61)` and we're filtering out games with pass moves) and `int` format (i.e. 0 to 63 inclusive, since the games contain moves from `A0` to `H7`).

In [ ]:
board_seqs_id = t.from_numpy(np.load(section_dir / "board_seqs_id_small.npy")).long()
board_seqs_square = t.from_numpy(np.load(section_dir / "board_seqs_square_small.npy")).long()

print(f"board_seqs_id: shape {tuple(board_seqs_id.shape)}, range: {board_seqs_id.min()} to {board_seqs_id.max()}")
print(
    f"board_seqs_square: shape {tuple(board_seqs_square.shape)}, range: {board_seqs_square.min()} to {board_seqs_square.max()}"
)

Note - you can access a larger dataset at the [GitHub readme](https://github.com/likenneth/othello_world), in the "Training Othello-GPT" section. There are links to download datasets from Google Drive (both synthetic and championship games). You can store them in your `data` folder, after you've cloned the repo using the code above.

## Making some utilities

At this point, we'll stop and get some aggregate data that will be useful later - a tensor of valid moves, of board states, and a cache of all model activations across 50 games (in practice, you want as much as will comfortably fit into GPU memory). It's really convenient to have the ability to quickly run an experiment across a bunch of games! And one of the great things about small models on algorithmic tasks is that you just can do stuff like this.

Let's call these the **focus games**.

In [ ]:
def get_board_states_and_legal_moves(
    games_square: Int[Tensor, "n_games n_moves"],
) -> tuple[
    Int[Tensor, "n_games n_moves rows cols"],
    Int[Tensor, "n_games n_moves rows cols"],
    list,
]:
    """
    Returns the following:
        states:                 (n_games, n_moves, 8, 8): tensor of board states after each move
        legal_moves:            (n_games, n_moves, 8, 8): tensor of 1s for legal moves, 0s for
                                    illegal moves
        legal_moves_annotation: (n_games, n_moves, 8, 8): list containing strings of "o" for legal
                                    moves (for plotting)
    """
    # Create tensors to store the board state & legal moves
    n_games, n_moves = games_square.shape
    states = t.zeros((n_games, 60, 8, 8), dtype=t.int32)
    legal_moves = t.zeros((n_games, 60, 8, 8), dtype=t.int32)

    # Loop over each game, populating state & legal moves tensors after each move
    for n in range(n_games):
        board = utils.OthelloBoardState()
        for i in range(n_moves):
            board.umpire(games_square[n, i].item())
            states[n, i] = t.from_numpy(board.state)
            legal_moves[n, i].flatten()[board.get_valid_moves()] = 1

    # Convert legal moves to annotation
    legal_moves_annotation = np.where(to_numpy(legal_moves), "o", "").tolist()

    return states, legal_moves, legal_moves_annotation


num_games = 50

focus_games_id = board_seqs_id[:num_games]  # shape [50, 60]
focus_games_square = board_seqs_square[:num_games]  # shape [50, 60]
focus_states, focus_legal_moves, focus_legal_moves_annotation = get_board_states_and_legal_moves(focus_games_square)

print("focus states:", focus_states.shape)
print("focus_legal_moves", tuple(focus_legal_moves.shape))

# Plot the first 10 moves of the first game
utils.plot_board_values(
    focus_states[0, :10],
    title="Board states",
    width=1000,
    boards_per_row=5,
    board_titles=[f"Move {i}, {'white' if i % 2 == 1 else 'black'} to play" for i in range(1, 11)],
    text=np.where(to_numpy(focus_legal_moves[0, :10]), "o", "").tolist(),
)

Let's also cache all the model's activations and logits for these focus games.

In [ ]:
focus_logits, focus_cache = model.run_with_cache(focus_games_id[:, :-1].to(device))

print(focus_logits.shape)  # shape [num_games=50, n_ctx=59, d_vocab=61]

> #### Recap of the useful objects we've defined
>
> We have the following:
> 
> Models
> - `model` is an 8-layer autoregressive transformer, trained to predict legal Othello moves. Its vocab is `range(0, 61)` where 0 = "pass" and the other numbers represent the 60 possible moves, excluding the 4 middle squares
>
> All data
> - `board_seqs_id`, shape `(100k, 60)` contains the moves from all 100k games (as token ids)
> - `board_seqs_square`, shape `(100k, 60)` contains the moves from all 100k games (as ints)
> 
> Focus games data
> - `focus_games_id`, shape `(50, 60)` contains the moves from 50 games (as token ids)
> - `focus_games_square`, shape `(50, 60)` contains the moves from 50 games (as ints)
> - `focus_states`, shape `(50, 60, 8, 8)` contains the board states after each move (0 = empty, 1 = black, -1 = white)
> - `focus_legal_moves`, shape `(50, 60, 8, 8)` contains a 1 for each legal move, and 0 for each illegal move
> - `focus_logits`, shape `(50, 59, 61)` contains model's output logits on focus games - `59=model.cfg.n_ctx` because we don't include the final move in our forward pass, and `61=model.cfg.d_vocab` contains the "pass" move and all 60 playable squares

## What is a probe?

From the [MI Dynalist notes](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#q=probe):

> **[Probing](https://arxiv.org/pdf/1610.01644.pdf)** is a technique for identifying directions in network activation space that correspond to a concept/feature.
>
> In spirit, you give the network a bunch of inputs with that feature, and a bunch without it. You train a linear map on a specific activation (eg the output of layer 5) which distinguishes these two sets, giving a 1D linear map (a **probe**), corresponding to a direction in activation space, which likely corresponds to that feature.

Probes can be a very valuable tool to help us better understand the concepts represented in our model. However, there are two big caveats to keep in mind:

1. Probes give us a direction, but they don't give us a causal story about how that direction got into the model in the first place, or how the model is using that direction.
2. Probes (especially nonlinear probes) can be hiding a lot of computation under their surface.

In the original paper analysing Othello, the authors used nonlinear probing to find important directions. This went against a fundamental intuition - that models fundamentally store things in linear ways, and so we should be able to access them with linear probes. In these exercises, we'll be using linear probes.

## Using the probe

The training of this probe was kind of a mess, and I'd do a bunch of things differently if doing it again.

There were 3 different probe modes:

- `full_linear_probe[0].shape = (d_model, 8, 8, 3)` was trained on black to play, i.e. odd moves. The classes are `[empty, white, black]`.
- `full_linear_probe[1].shape = (d_model, 8, 8, 3)` was trained on white to play, i.e. even moves. The classes are `[empty, white, black]`.
- `full_linear_probe[2].shape = (d_model, 8, 8, 3)` was trained on all moves.

For example, we could take `full_linear_probe[0]` and take the inner product with the residual stream to get a tensor of shape `(8, 8, 3)` representing `8x8=64` separate predictions for what each of the 64 board squares contains (i.e. you'd softmax this tensor over the last dimension to get probabilities).

In [ ]:
full_linear_probe = t.load(section_dir / "main_linear_probe.pth", map_location=str(device), weights_only=True)

print(full_linear_probe.shape)

# Define indices along `full_linear_probe.shape[0]`, i.e. the different probe modes
black_to_play, white_to_play, _ = (0, 1, 2)
# Define indices along `full_linear_probe.shape[-1]`, i.e. the different classifications for each mode
empty, white, black = (0, 1, 2)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "1.5.3.15"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part53_othellogpt.solutions import plot_probe_outputs, apply_scale, calculate_attn_and_mlp_probe_score_contributions, get_w_in, get_w_in, patching_metric, patch_final_move_output, make_spectrum_plot, ProbeTrainingArgs, ProbeTrainingArgs


### Exercise - fill in the missing code below

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 30-40 minutes on this exercise.
> There are several steps to this exercise, so after trying for some time you're recommended to look at the solution.
> ```

We've just left the `training_step` function incomplete, so that's the one you need to fill in. It should return the loss which you backpropagate on (you can see the code which actually performs the backprop algorithm below).

To make this exercise easier we're only having you train a single probe mode, on even moves (as a bonus exercise you can try training 3 modes at once - if you do this then remember to flip the labels for odd and even moves when you compute loss for the "even & odd" probe mode).

Your `training_step` function should:

- Use `model.run_with_cache` to get the cached residual stream values for the layer you're training on. 
    - Tip: you can use the `stop_at_layer` argument to prevent unnecessary computation
    - Tip: you can also the `names_filter` argument to only get the activations you need (this can be a hook name or a function mapping hook names to bools)
    - Tip: remember to use inference mode here, since we're only running our model to get the activations to feed into our probe.
    - Tip: remember that the `games_id` tensor you're given has shape `(batch_size, 60)`, so you'll need to slice off the last game before passing into the model.
- Slice the activations appropriately (we have `args.pos_start` and `args.pos_end` telling you which positions to use, and also we're only using the even positions for now)
- Compute the probe output, by taking an inner product over the `d_model` dimension and summing over positions.
- Convert the probe output to logprobs, and get correct logprobs by indexing into it.
    - Tip: we're using even positions, so remember that we convert from board state to our probe basis using "mine=black, theirs=white".
    - Tip: this indexing will require taking code from earlier which computes board state from the game sequence. We recommend just using `get_board_states_and_legal_moves` and keeping the `state` object returned - remember that this state object has values 0, 1, 2 corresponding to options empty, white, black.
- Compute the loss, by summing over the row & column dimensions and averaging over the batch & seqpos dimensions (this is because the latter are data batch dimensions, but the former are effectively probe batch dimensions, since we're training multiple probes at once on each board square).
- Return the loss (this code is given to you).

Once you've finished implementing this function, you can run the code below. Your training loss should quickly drop below `64 * ln(3) ≈ 70` (which is the loss you'd get from uniform random guesses about what occupies each square). This loss should fall to around 10-20 by the end of training, if you use the default hyperparameters.

We recommend you set `use_wandb=False` while working on this code, until you're getting it running without errors.

In [ ]:
class LinearProbeTrainer:
    def __init__(self, model: HookedTransformer, args: ProbeTrainingArgs):
        self.model = model
        self.args = args
        self.linear_probe = args.setup_linear_probe(model)

    def training_step(self, indices: Int[Tensor, "n_games"]) -> Float[Tensor, ""]:
        # Use indices to slice our batch of games (remember, games_id = token IDs
        # from 1 to 60, and games_square = indices of squares in board)
        indices_cpu = indices.cpu()
        games_id = board_seqs_id[indices_cpu]  # shape [batch n_moves=60]
        games_square = board_seqs_square[indices_cpu]  # shape [batch n_moves=60]

        # Define seqpos slicing (note, we add n_ctx to pos_end to deal with the zero case)
        pos_start = self.args.pos_start
        pos_end = self.args.pos_end + self.model.cfg.n_ctx

        # YOUR CODE HERE - define loss

        if self.args.use_wandb:
            wandb.log(dict(loss=loss.item()), step=self.step)
        self.step += 1

        return loss

    def shuffle_training_indices(self):
        """
        Returns the tensors you'll use to index into the training data.
        """
        n_indices = self.args.num_games - (self.args.num_games % self.args.batch_size)
        full_train_indices = t.randperm(self.args.num_games)[:n_indices]
        full_train_indices = einops.rearrange(
            full_train_indices,
            "(batch_idx game_idx) -> batch_idx game_idx",
            game_idx=self.args.batch_size,
        )
        return full_train_indices

    def train(self):
        self.step = 0
        if self.args.use_wandb:
            wandb.init(project=self.args.wandb_project, name=self.args.wandb_name, config=self.args)

        optimizer = t.optim.AdamW(
            [self.linear_probe],
            lr=self.args.lr,
            betas=self.args.betas,
            weight_decay=self.args.weight_decay,
        )

        for epoch in range(self.args.epochs):
            print(f"Epoch {epoch + 1}/{self.args.epochs}")
            full_train_indices = self.shuffle_training_indices()
            progress_bar = tqdm(full_train_indices)
            for indices in progress_bar:
                loss = self.training_step(indices)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                progress_bar.set_description(f"Loss = {loss:.4f}")

        if self.args.use_wandb:
            wandb.finish()


t.set_grad_enabled(True)

args = ProbeTrainingArgs()
trainer = LinearProbeTrainer(model, args)
trainer.train()

<details><summary>Solution</summary>

```python
class LinearProbeTrainer:
    def __init__(self, model: HookedTransformer, args: ProbeTrainingArgs):
        self.model = model
        self.args = args
        self.linear_probe = args.setup_linear_probe(model)

    def training_step(self, indices: Int[Tensor, "n_games"]) -> Float[Tensor, ""]:
        # Use indices to slice our batch of games (remember, games_id = token IDs
        # from 1 to 60, and games_square = indices of squares in board)
        indices_cpu = indices.cpu()
        games_id = board_seqs_id[indices_cpu]  # shape [batch n_moves=60]
        games_square = board_seqs_square[indices_cpu]  # shape [batch n_moves=60]

        # Define seqpos slicing (note, we add n_ctx to pos_end to deal with the zero case)
        pos_start = self.args.pos_start
        pos_end = self.args.pos_end + self.model.cfg.n_ctx

        # Cache resid_post from all our games (ignoring the last one)
        with t.inference_mode():
            _, cache = model.run_with_cache(
                games_id[:, :-1].to(device),
                return_type=None,
                names_filter=lambda name: name.endswith("resid_post"),
            )

        # We slice from the first even index (on or after pos_start), since we're
        # just looking at predictions made after white has played a move.
        pos_start_even = pos_start + (pos_start % 2)
        seqpos_indices = np.arange(pos_start_even, pos_end, 2)
        resid_post = cache["resid_post", self.args.layer][:, seqpos_indices]

        # Get probe output, i.e. probe_logits[g, p, r, c] = the 3-vector of logit
        # predictions for what color is in square [r, c] AFTER the p-th move is
        # played in game g.
        probe_logits = einops.einsum(
            resid_post,
            self.linear_probe,
            "batch pos d_model, d_model rows cols options -> batch pos rows cols options",
        )
        probe_logprobs = probe_logits.log_softmax(-1)

        # Get the actual game state. The original state has {0: empty, 1: black, -1: white} and
        # we want our probe to be in the basis {0: empty, 1: theirs, 2: mine}. We're only training
        # on even moves i.e. black just played and mine = white, so we just need to map -1 -> 2.
        state = get_board_states_and_legal_moves(games_square)[0]  # shape [batch moves 8 8]
        state = state[:, seqpos_indices]  # shape [batch pos 8 8]
        state[state == -1] = 2

        # Index into probe logprobs to get the logprobs for correct board state, and then
        # return loss as the mean over games & posns, and sum over rows & cols (since each
        # row & col is effectively an independent probe).
        correct_probe_logprobs = eindex(probe_logprobs, state, "game pos row col [game pos row col]")
        loss = -einops.reduce(correct_probe_logprobs, "game pos row col -> row col", "mean").sum()

        if self.args.use_wandb:
            wandb.log(dict(loss=loss.item()), step=self.step)
        self.step += 1

        return loss

    def shuffle_training_indices(self):
        """
        Returns the tensors you'll use to index into the training data.
        """
        n_indices = self.args.num_games - (self.args.num_games % self.args.batch_size)
        full_train_indices = t.randperm(self.args.num_games)[:n_indices]
        full_train_indices = einops.rearrange(
            full_train_indices,
            "(batch_idx game_idx) -> batch_idx game_idx",
            game_idx=self.args.batch_size,
        )
        return full_train_indices

    def train(self):
        self.step = 0
        if self.args.use_wandb:
            wandb.init(project=self.args.wandb_project, name=self.args.wandb_name, config=self.args)

        optimizer = t.optim.AdamW(
            [self.linear_probe],
            lr=self.args.lr,
            betas=self.args.betas,
            weight_decay=self.args.weight_decay,
        )

        for epoch in range(self.args.epochs):
            print(f"Epoch {epoch + 1}/{self.args.epochs}")
            full_train_indices = self.shuffle_training_indices()
            progress_bar = tqdm(full_train_indices)
            for indices in progress_bar:
                loss = self.training_step(indices)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                progress_bar.set_description(f"Loss = {loss:.4f}")

        if self.args.use_wandb:
            wandb.finish()
```
</details>

Finally, let's make the same accuracy plot from before, and see how well it works. Note that we're not constructing a new probe by averaging the even and odd mode probes this time, we're just taking our single even mode probe and using it (there's a bonus exercise below where you can train all 3 modes at once).

In [ ]:
# Getting the probe's output, and then its predictions
probe_out = einops.einsum(
    focus_cache["resid_post", args.layer],
    trainer.linear_probe,
    "game move d_model, d_model row col options -> game move row col options",
)
probe_out_value = probe_out.argmax(dim=-1).cpu()

# See what the accuracy was in 3 cases: odd moves, even moves, and aggregate moves
is_correct = probe_out_value == focus_states_theirs_vs_mine[:, :-1]
accuracies_odd = einops.reduce(is_correct[:, 5:-5:2].float(), "game move row col -> row col", "mean")
accuracies_even = einops.reduce(is_correct[:, 6:-6:2].float(), "game move row col -> row col", "mean")
accuracies_all = einops.reduce(is_correct[:, 5:-5].float(), "game move row col -> row col", "mean")

utils.plot_board_values(
    1 - t.stack([accuracies_odd, accuracies_even, accuracies_all], dim=0),
    title="Average Error Rate of Linear Probe",
    board_titles=["Black to play", "White to play", "All Moves"],
    zmax=0.25,
    zmin=-0.25,
    height=400,
    width=900,
)

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
